# PyADM1ODE: Complex Two-Stage Plant Example

This notebook demonstrates a two-stage biogas plant with mechanical components (pumps, mixers) and energy integration (CHP, heating).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgaida/PyADM1ODE/blob/master/examples/colab_02_complex_plant.ipynb)

## 1. Setup Environment

In [ ]:
# Install Mono and other dependencies
!apt-get update
!apt-get install -y mono-devel

# Clone the repository and install pyadm1
!git clone https://github.com/dgaida/PyADM1ODE.git
%cd PyADM1ODE
!pip install -e .

## 2. Import Libraries

In [ ]:
from pyadm1.configurator.plant_builder import BiogasPlant
from pyadm1.substrates.feedstock import Feedstock
from pyadm1.configurator.plant_configurator import PlantConfigurator

## 3. Configure and Run Simulation

In [ ]:
initial_state_file = "data/initial_states/digester_initial8.csv"

# 1. Create feedstock
feedstock = Feedstock(feeding_freq=48)
plant = BiogasPlant("Two-Stage Plant")
configurator = PlantConfigurator(plant, feedstock)

# 2. Add Stage 1 (Hydrolysis)
configurator.add_digester(
    digester_id="digester_1",
    V_liq=1977.0,
    V_gas=304.0,
    T_ad=318.15,  # 45°C
    load_initial_state=True,
    initial_state_file=initial_state_file,
    Q_substrates=[15, 10, 0, 0, 0, 0, 0, 0, 0, 0]
)

# 3. Add Stage 2 (Methanogenesis)
configurator.add_digester(
    digester_id="digester_2",
    V_liq=1000.0,
    V_gas=150.0,
    T_ad=308.15,  # 35°C
    load_initial_state=True,
    initial_state_file=initial_state_file,
    Q_substrates=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
)

# 4. Add CHP and Heating
configurator.add_chp("chp_1", P_el_nom=500.0)
configurator.add_heating("heating_1", target_temperature=318.15)
configurator.add_heating("heating_2", target_temperature=308.15)

# 5. Connect components
configurator.connect("digester_1", "digester_2", "liquid")
configurator.auto_connect_digester_to_chp("digester_1", "chp_1")
configurator.auto_connect_digester_to_chp("digester_2", "chp_1")
configurator.auto_connect_chp_to_heating("chp_1", "heating_1")
configurator.auto_connect_chp_to_heating("chp_1", "heating_2")

# 6. Initialize and simulate
plant.initialize()
results = plant.simulate(duration=5.0, dt=1.0/24.0, save_interval=1.0)

print("\nSimulation completed successfully!")

## 4. Analyze Results

In [ ]:
final = results[-1]["components"]
print(f"Total Biogas:  {final['digester_1']['Q_gas'] + final['digester_2']['Q_gas']:.1f} m³/d")
print(f"Total Methane: {final['digester_1']['Q_ch4'] + final['digester_2']['Q_ch4']:.1f} m³/d")
print(f"CHP Power:     {final['chp_1']['P_el']:.1f} kW")